In [3]:
import numpy as np
import pandas as pd
from scipy.stats import skewnorm
from scipy.stats import pearsonr, spearmanr
from scipy.stats import norm
from scipy.spatial.distance import euclidean
import statsmodels.api as sm
import os
import re
import time
import gc
import random
from tools import *
random.seed(621)

In [4]:
output_folder = r"Z:\Projects\EMA_Project\Scripts\Output\Simulation2_Scratch"
results_folder = r"Z:\Projects\EMA_Project\Scripts\Output\Simulation2_Results"
os.makedirs(output_folder, exist_ok=True)
os.makedirs(results_folder, exist_ok=True)
independent_vars = ["X"]

# Define B1, B2
LB1 = 1   # Coefficient for x
QB2 = 0.7   # Coefficient for x^2

n_subjects = 105
n_timepoints = 65
n_simulations = 1000
states = [1, 2, 3, 4, 5]

In [5]:
def normalize_transition_matrix(matrix):
    norm_matrix = np.zeros_like(matrix)
    for i, row in enumerate(matrix):
        row_sum = np.sum(row)
        if row_sum == 0:
            norm_matrix[i] = np.ones_like(row) / len(row)
        else:
            norm_matrix[i] = row / row_sum
    return norm_matrix

In [6]:
for sim in range(n_simulations):
    # Generate random transition matrices for each subject
    X_transition_matrices = []
    Y_transition_matrices = []
    for _ in range(n_subjects):
        X_mat = np.random.uniform(0, 1, (5, 5))
        X_mat = X_mat / X_mat.sum(axis=1, keepdims=True)
        X_transition_matrices.append(X_mat)

        Y_mat = np.random.uniform(0, 1, (5, 5))
        Y_mat = Y_mat / Y_mat.sum(axis=1, keepdims=True)
        Y_transition_matrices.append(Y_mat)

    print(f"X: {len(X_transition_matrices)} generated")
    print(f"Y: {len(Y_transition_matrices)} generated")

    for iv in independent_vars:
        print(f"Running simulations for {iv}...")

        transition_matrix_dict = {
            "X": X_transition_matrices
        }

        iv_matrices = transition_matrix_dict[iv]
        Y_matrices = Y_transition_matrices

        walk_results = []

        for i in range(n_subjects):
            iv_mat = iv_matrices[i]
            Y_mat = Y_matrices[i]

            # (1) Null: y is generated using Y matrix from a different subject (should not actually matter given that they are generated)
            j = (i + np.random.randint(1, n_subjects)) % n_subjects
            alt_Y_mat = Y_matrices[j]

            # Random walk for x(t) using IV matrix
            x = [random.choice(states)]
            for t in range(1, n_timepoints):
                row = iv_mat[x[-1] - 1]
                x.append(np.random.choice(states, p=row))

            # Random walk for y(t) using Y matrix (null)
            y_null = [random.choice(states)]
            for t in range(1, n_timepoints):
                row = alt_Y_mat[y_null[-1] - 1]
                y_null.append(np.random.choice(states, p=row))

            # Polynomial models
            y_poly1 = []
            y_poly2 = []
            for t in range(n_timepoints):
                y_poly1.append(np.clip(int(LB1 * (x[t])), 1, 5))
                y_poly2.append(np.clip(int(np.round((1 + (QB2 * ((x[t]-3) ** 2))))), 1, 5))

            # Save long format: one row per timepoint
            walk_df = pd.DataFrame({
                "subject": i,
                "time": np.arange(n_timepoints),
                "iv_walk": x,
                "y_null": y_null,
                "y_poly1": y_poly1,
                "y_poly2": y_poly2,
            })
            filename = f"{iv}__sim_{sim}_sub_{i}.csv"
            walk_df.to_csv(os.path.join(output_folder, filename), index=False)

X: 105 generated
Y: 105 generated
Running simulations for X...
X: 105 generated
Y: 105 generated
Running simulations for X...
X: 105 generated
Y: 105 generated
Running simulations for X...
X: 105 generated
Y: 105 generated
Running simulations for X...
X: 105 generated
Y: 105 generated
Running simulations for X...
X: 105 generated
Y: 105 generated
Running simulations for X...
X: 105 generated
Y: 105 generated
Running simulations for X...
X: 105 generated
Y: 105 generated
Running simulations for X...
X: 105 generated
Y: 105 generated
Running simulations for X...
X: 105 generated
Y: 105 generated
Running simulations for X...
X: 105 generated
Y: 105 generated
Running simulations for X...
X: 105 generated
Y: 105 generated
Running simulations for X...
X: 105 generated
Y: 105 generated
Running simulations for X...
X: 105 generated
Y: 105 generated
Running simulations for X...
X: 105 generated
Y: 105 generated
Running simulations for X...
X: 105 generated
Y: 105 generated
Running simulations f

In [7]:
definition_transition_matrix2 = ["Transition", "Proportion"]
distance_measure = ["Pearson", "Spearman", "Euclid"]
comparison_method = ["Pairwise", "Leave-One-Subject-Out"]
y_relationships = ["y_null", "y_poly1", "y_poly2"]

# Loop through each of the 1000 simulations and then loop through each of the versions of Y within that
# Create pTS which is a row for each subject for a given simulation and then their x's at each time point
# Create dvTS which is a row for each subject for a given simulation and y relationship and then their y's at each time point
# Do first-order regression for the simulation+y relationship we are on and then do the multiverse version, make sure we cluster (sm.OLS) standard errors on subject
for iv in independent_vars:
    for sim in range(n_simulations):
        print(f"\nProcessing simulation {sim}")

        for dv_rel in y_relationships:
            # Initialize storage for predictor and DV time series
            pTS = []
            dvTS = []

            for i in range(n_subjects):
                file = os.path.join(
                    output_folder,
                    f"{iv}__sim_{sim}_sub_{i}.csv"
                )
                if not os.path.exists(file):
                    print(f"Missing: {file}")
                    continue

                df = pd.read_csv(file)
                pTS.append(df["iv_walk"].values.tolist())
                dvTS.append(df[dv_rel].values.tolist())

            # Turn lists of lists into DataFrames with subject index
            pTS_df = pd.DataFrame(pTS)
            dvTS_df = pd.DataFrame(dvTS)

            # Make sure index corresponds to subject ID
            pTS_df.index = list(range(len(pTS)))
            dvTS_df.index = list(range(len(dvTS)))

            # Melt to long format for regression (one row per subject-timepoint)
            pTS_long = pTS_df.reset_index().melt(id_vars="index", var_name="time", value_name="iv")
            dvTS_long = dvTS_df.reset_index().melt(id_vars="index", var_name="time", value_name="dv")
            long_df = pd.merge(pTS_long, dvTS_long, on=["index", "time"])
            long_df.rename(columns={"index": "subject"}, inplace=True)
            long_df.dropna(inplace=True)

            # Skip if not enough subjects for clustering
            if long_df["subject"].nunique() < 2:
                print(f"Not enough subjects for clustered SEs in simulation {sim}, {dv_rel}. Skipping.")
                continue

            # Run first-order regression with subject-clustered SEs
            X = sm.add_constant(long_df["iv"])
            y = long_df["dv"]
            model = sm.OLS(y, X).fit(cov_type="cluster", cov_kwds={"groups": long_df["subject"]})

            # Only keep the slope (parameter for independent variable)
            slope = model.params[1]
            slope_se = model.bse[1]
            slope_pval = model.pvalues[1]
            obs = int(model.nobs)

            multiverse_results = []

            for definition in definition_transition_matrix2:
                if definition == "Transition":
                    P_timeseries_list, P_transition_matrices = indiv_ts_tmat(pTS_df)
                    DV_timeseries_list, DV_transition_matrices = indiv_ts_tmat(dvTS_df)
                if definition == "Proportion":
                    P_timeseries_list, P_transition_matrices = indiv_ts_pmat(pTS_df)
                    DV_timeseries_list, DV_transition_matrices = indiv_ts_pmat(dvTS_df)

                for measure in distance_measure:
                    if measure == "Spearman":
                        use = "spear"
                    elif measure == "Pearson":
                        use = "pearson"
                    elif measure == "Euclid":
                        use = "euclid"
                    for comparison in comparison_method:
                        if comparison == "Leave-One-Subject-Out":
                            comp = "loso_similarity_matrix"
                        elif comparison == "Pairwise":
                            comp = "compute_rsm"

                        function_name = f"{comp}_{use}"
                        print(f"Calling function: {function_name}")

                        # Calculate similarity matrices
                        Prsm = globals()[function_name](P_transition_matrices)
                        DVrsm = globals()[function_name](DV_transition_matrices)

                        if comparison == "Leave-One-Subject-Out":
                            if measure in ["Pearson", "Spearman"]:
                                Prdm = [1 - corr for corr in Prsm]
                                DVrdm = [1 - corr for corr in DVrsm]
                            elif measure == "Euclid":
                                Prdm = [2 * corr for corr in Prsm]
                                DVrdm = [2 * corr for corr in DVrsm]
                        elif comparison == "Pairwise":
                            if measure in ["Pearson", "Spearman"]:
                                Prdm = 1 - Prsm
                                DVrdm = 1 - DVrsm
                            elif measure == "Euclid":
                                Prdm = 2 * Prsm
                                DVrdm = 2 * DVrsm
                                Prdm = np.array(Prdm)
                            lower_triangle_indices = np.tril_indices_from(Prdm, k=-1)
                            Prdm = Prdm[lower_triangle_indices]
                            DVrdm = DVrdm[lower_triangle_indices]

                        # Add constant for regression
                        X_mv = sm.add_constant(Prdm)
                        y_mv = DVrdm
                        mv_model = sm.OLS(y_mv, X_mv).fit()

                        # Only keep the slope (parameter for Prdm)
                        mv_slope = mv_model.params[1]
                        mv_slope_se = mv_model.bse[1]
                        mv_slope_pval = mv_model.pvalues[1]
                        mv_obs = int(mv_model.nobs)

                        multiverse_results.append({
                            "Specification": f"{dv_rel}_{definition}_{measure}_{comparison}",
                            "Parameter": mv_slope,
                            "Std_Err": mv_slope_se,
                            "P_value": mv_slope_pval,
                            "Observations": mv_obs,
                            "FirstOrder_Param": slope,
                            "FirstOrder_StdErr": slope_se,
                            "FirstOrder_Pval": slope_pval,
                            "FirstOrder_Obs": obs
                        })

            # Save all multiverse results for this (iv, sim, dv_rel)
            results_df = pd.DataFrame(multiverse_results)
            specification = f"{dv_rel}_{iv}_sim_{sim}"
            results_df.to_csv(os.path.join(results_folder, f"Sim_result_{specification}.csv"), index=False)
            print(f"Processed Sim_result_{specification}")


Processing simulation 0


C:\Users\CDN_Lab\AppData\Local\Temp\ipykernel_31340\1507421974.py:58: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slope = model.params[1]
C:\Users\CDN_Lab\AppData\Local\Temp\ipykernel_31340\1507421974.py:59: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slope_se = model.bse[1]
C:\Users\CDN_Lab\AppData\Local\Temp\ipykernel_31340\1507421974.py:60: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slope_pval = model.pvalues[1]


Calling function: compute_rsm_pearson
Calling function: loso_similarity_matrix_pearson
Calling function: compute_rsm_spear
Calling function: loso_similarity_matrix_spear
Calling function: compute_rsm_euclid
Calling function: loso_similarity_matrix_euclid
Calling function: compute_rsm_pearson
Calling function: loso_similarity_matrix_pearson
Calling function: compute_rsm_spear
Calling function: loso_similarity_matrix_spear
Calling function: compute_rsm_euclid
Calling function: loso_similarity_matrix_euclid
Processed Sim_result_y_null_X_sim_0


C:\Users\CDN_Lab\AppData\Local\Temp\ipykernel_31340\1507421974.py:58: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slope = model.params[1]
C:\Users\CDN_Lab\AppData\Local\Temp\ipykernel_31340\1507421974.py:59: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slope_se = model.bse[1]
C:\Users\CDN_Lab\AppData\Local\Temp\ipykernel_31340\1507421974.py:60: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slope_pval = model.pvalues[1]


Calling function: compute_rsm_pearson
Calling function: loso_similarity_matrix_pearson
Calling function: compute_rsm_spear
Calling function: loso_similarity_matrix_spear
Calling function: compute_rsm_euclid
Calling function: loso_similarity_matrix_euclid
Calling function: compute_rsm_pearson
Calling function: loso_similarity_matrix_pearson
Calling function: compute_rsm_spear
Calling function: loso_similarity_matrix_spear
Calling function: compute_rsm_euclid
Calling function: loso_similarity_matrix_euclid
Processed Sim_result_y_poly1_X_sim_0


...

Calling function: compute_rsm_pearson
Calling function: loso_similarity_matrix_pearson
Calling function: compute_rsm_spear
Calling function: loso_similarity_matrix_spear
Calling function: compute_rsm_euclid
Calling function: loso_similarity_matrix_euclid
Calling function: compute_rsm_pearson
Calling function: loso_similarity_matrix_pearson
Calling function: compute_rsm_spear
Calling function: loso_similarity_matrix_spear
Calling function: compute_rsm_euclid
Calling function: loso_similarity_matrix_euclid
Processed Sim_result_y_null_X_sim_999


C:\Users\CDN_Lab\AppData\Local\Temp\ipykernel_31340\1507421974.py:58: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slope = model.params[1]
C:\Users\CDN_Lab\AppData\Local\Temp\ipykernel_31340\1507421974.py:59: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slope_se = model.bse[1]
C:\Users\CDN_Lab\AppData\Local\Temp\ipykernel_31340\1507421974.py:60: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slope_pval = model.pvalues[1]


Calling function: compute_rsm_pearson
Calling function: loso_similarity_matrix_pearson
Calling function: compute_rsm_spear
Calling function: loso_similarity_matrix_spear
Calling function: compute_rsm_euclid
Calling function: loso_similarity_matrix_euclid
Calling function: compute_rsm_pearson
Calling function: loso_similarity_matrix_pearson
Calling function: compute_rsm_spear
Calling function: loso_similarity_matrix_spear
Calling function: compute_rsm_euclid
Calling function: loso_similarity_matrix_euclid
Processed Sim_result_y_poly1_X_sim_999


C:\Users\CDN_Lab\AppData\Local\Temp\ipykernel_31340\1507421974.py:58: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slope = model.params[1]
C:\Users\CDN_Lab\AppData\Local\Temp\ipykernel_31340\1507421974.py:59: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slope_se = model.bse[1]
C:\Users\CDN_Lab\AppData\Local\Temp\ipykernel_31340\1507421974.py:60: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slope_pval = model.pvalues[1]


Calling function: compute_rsm_pearson
Calling function: loso_similarity_matrix_pearson
Calling function: compute_rsm_spear
Calling function: loso_similarity_matrix_spear
Calling function: compute_rsm_euclid
Calling function: loso_similarity_matrix_euclid
Calling function: compute_rsm_pearson
Calling function: loso_similarity_matrix_pearson
Calling function: compute_rsm_spear
Calling function: loso_similarity_matrix_spear
Calling function: compute_rsm_euclid
Calling function: loso_similarity_matrix_euclid
Processed Sim_result_y_poly2_X_sim_999
